<a href="https://colab.research.google.com/github/prannav-vg/VLM_fun/blob/main/Video_VLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U transformers accelerate qwen-vl-utils av

import os, subprocess
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from google.colab import files

model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id)

uploaded = files.upload()
original_path = os.path.abspath(next(iter(uploaded)))

small_path = "/content/video_small.mp4"
subprocess.run([
    "ffmpeg", "-y", "-i", original_path,
    "-vf", "scale=-2:480", "-r", "10",
    "-c:v", "libx264", "-preset", "veryfast", "-an",
    small_path
], check=True, capture_output=True)
print("Converted to 480p:", small_path)

def ask_video(question, path=small_path, fps=1.0):
    messages = [{
        "role": "user",
        "content": [
            {"type": "video", "video": path, "max_pixels": 360 * 420, "fps": fps},
            {"type": "text", "text": question},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256)
    out = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(out, skip_special_tokens=True)[0]

print(ask_video("Describe this video in detail."))

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Saving Video Project 17.mp4 to Video Project 17.mp4
Converted to 480p: /content/video_small.mp4


qwen-vl-utils using torchcodec to read video.
[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


In the video, we see a group of people working in a modern office setting. The room is well-lit with natural light coming in through large windows. The office is spacious and has a contemporary design with black furniture and a few plants adding a touch of greenery. 

The main focus of the video is on two men sitting at their desks. One of the men is holding a cup of coffee and appears to be deep in thought, possibly working on a project. The other man is using a laptop and appears to be focused on his work. 

As the video progresses, a third person enters the scene and sits down next to the man on the couch. The third person is also using a laptop and appears to be working on a project. The man on the couch looks up and smiles at the third person, indicating a friendly interaction between the two.

The office environment is busy and productive, with people working on various tasks. The lighting is bright and the space is clean and organized, creating a conducive atmosphere for work. T